# Compare iSSF (linear/MLP) vs deepSSF

Minimal notebook to:
- load checkpoints per study for `issf_linear`, `issf_mlp`, `deepssf`,
- run inference on **all test steps**,
- sample from each model’s probability surface:
  - deepSSF: 500 samples from the 101×101 grid,
  - iSSF: 10,000 candidates → 1,000 samples from candidate softmax.

Outputs (under `<output_dir>/compare_outputs/`):
- `summary.csv`
- `surface_samples.csv`


In [ ]:
from __future__ import annotations

import hashlib, math, sys
from pathlib import Path

import numpy as np, pandas as pd, torch

BENCHMARK_CONFIG_PATH = "../configs/benchmarks/all.yaml"
ONLY_ONE_STUDY = True
STUDY_ID: str | None = None  # e.g. "tsavo_lions"; if None uses first resolved study
SETUP_IF_MISSING = False
CHECKPOINT_ROOT_OVERRIDE: str | None = None

ISSF_SURFACE_NUM_CANDIDATES = 10_000
ISSF_SURFACE_NUM_SAMPLES = 1_000
DEEPSSF_SURFACE_NUM_SAMPLES = 500
SURFACE_STEPS_PER_STUDY: int | None = 10
SURFACE_SEED = 0
TOPK = (1, 5, 10, 25, 50)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def repo_root() -> Path:
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "movebench").is_dir():
            return q
    raise FileNotFoundError("Could not find repo root")


REPO_ROOT = repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def rpath(p: str | Path) -> Path:
    p = Path(p)
    if p.exists():
        return p.resolve()
    for base in (REPO_ROOT, REPO_ROOT / "notebooks"):
        q = (base / p).resolve()
        if q.exists():
            return q
    raise FileNotFoundError(p)


def seed(*parts: object) -> int:
    s = "|".join(str(p) for p in parts).encode("utf-8")
    return int(hashlib.sha256(s).hexdigest()[:8], 16)


print("REPO_ROOT:", REPO_ROOT)
print("DEVICE:", DEVICE)


In [ ]:
from __future__ import annotations

from tqdm.auto import tqdm
from torch.utils.data import DataLoader

from movebench import Study
from movebench import benchmark as mb_benchmark
from movebench.datasets.deepssf import DeepSSFDataset
from movebench.features.normalization import load_points_normalization_mean_std
from movebench.features.points import select_point_store_dir
from movebench.models.deepssf import DeepSSF
from movebench.models.ssf import LinearStepSelect, MLPStepSelect
from movebench.train import collate_deepssf, collate_step_select, eval_deepssf, eval_step_select
from movebench.trajectories import SET_COL, TIME_COL, compute_movement_features, select_step_starts

import movebench.features.points as mb_points


cfg_path = rpath(BENCHMARK_CONFIG_PATH)
cfg = mb_benchmark.load_config(cfg_path)
methods = {str(m["kind"]): dict(m) for m in cfg["methods"]}
for k in ("issf_linear", "issf_mlp", "deepssf"):
    if k not in methods:
        raise ValueError(f"Benchmark config missing {k}")

layers_config = rpath(cfg["layers_config"])
studies = mb_benchmark.resolve_studies(cfg)
if ONLY_ONE_STUDY:
    sid = STUDY_ID or str(studies[0])
    studies = [sid]

out_dir = Path(str(cfg["output_dir"]))
out_dir = out_dir if out_dir.is_absolute() else (REPO_ROOT / out_dir).resolve()
out_cmp = out_dir / "compare_outputs"
out_cmp.mkdir(parents=True, exist_ok=True)
ckpt_root = Path(CHECKPOINT_ROOT_OVERRIDE) if CHECKPOINT_ROOT_OVERRIDE else (out_dir / "checkpoints")

print("cfg:", cfg_path)
print("layers_config:", layers_config)
print("output_dir:", out_dir)
print("checkpoint_root:", ckpt_root)
print("n_studies:", len(studies), studies)
if not ckpt_root.exists():
    print("\nWARNING: no checkpoints found; run:")
    print(f"  python {REPO_ROOT/'benchmark.py'} --config {cfg_path}")


def ckpt_path(study_id: str, kind: str) -> Path:
    return ckpt_root / study_id / f"{methods[kind]['name']}.pt"


def load_model(study_id: str, kind: str) -> tuple[torch.nn.Module, dict[str, Any]] | None:
    p = ckpt_path(study_id, kind)
    if not p.exists():
        return None
    d = torch.load(p, map_location="cpu")
    meta = dict(d.get("meta", {}) or {})
    state = dict(d.get("state_dict", {}) or {})
    if kind == "issf_linear":
        m = LinearStepSelect(int(meta["in_channels"]))
    elif kind == "issf_mlp":
        mc = dict(methods[kind].get("model", {}) or {})
        hd = tuple(int(x) for x in mc.get("hidden_dims", (128,)))
        dr = float(mc.get("dropout", 0.6))
        m = MLPStepSelect(int(meta["in_channels"]), hidden_dims=hd, dropout=dr)
    elif kind == "deepssf":
        m = DeepSSF(int(meta["env_channels"]), image_dim=int(meta["image_dim"]), pixel_size_m=float(meta["pixel_size_m"]), device=DEVICE)
    else:
        raise ValueError(kind)
    m.load_state_dict(state)
    return m.to(DEVICE).eval(), meta


def get_sampler_hash(study: Study, plan_config_hash: str) -> str:
    s = cfg["datasets"]["issf"]["sampler"]
    try:
        _dir, m = select_point_store_dir(
            study_dir=study.study_dir,
            split="train",
            config_hash=str(plan_config_hash),
            expected_type="parametric",
            num_candidates=int(s["num_candidates"]),
            random_seed=int(s["random_seed"]),
            step_multiplier=float(s["step_multiplier"]),
            step_q=float(s["step_q"]),
            gap_q=float(s["gap_q"]),
            bounds_buffer_deg=float(s["bounds_buffer_deg"]),
        )
        return str(m.sampler_hash)
    except FileNotFoundError:
        if not SETUP_IF_MISSING:
            raise
        _ = mb_benchmark.ensure_study_ready(study=study, cfg=cfg)
        return get_sampler_hash(study, plan_config_hash)


def mv_stats_from_train_store(study: Study, plan_config_hash: str, sampler_hash: str) -> dict[str, float]:
    pt = study.study_dir / "features" / "points" / "train" / plan_config_hash / sampler_hash / "features.pt"
    d = torch.load(pt, map_location="cpu")
    sl = d["step_length_m"][:, -1].to(dtype=torch.float64)
    ta = d["turn_angle_rad"][:, -1].to(dtype=torch.float64)
    log_sl, cos_ta = torch.log(sl + 1e-8), torch.cos(ta)

    def ms(x: torch.Tensor) -> tuple[float, float]:
        m, s = float(x.mean().item()), float(x.std(unbiased=False).item())
        if not math.isfinite(s) or s < 1e-8:
            s = 1.0
        return m, s

    sl_m, sl_s = ms(sl)
    ls_m, ls_s = ms(log_sl)
    ct_m, ct_s = ms(cos_ta)
    return {"sl_mean": sl_m, "sl_std": sl_s, "log_sl_mean": ls_m, "log_sl_std": ls_s, "cos_ta_mean": ct_m, "cos_ta_std": ct_s}


def issf_step_x(env: torch.Tensor, valid: torch.Tensor, sl_m: torch.Tensor, ta: torch.Tensor, tsn: int, mv: dict[str, float], mean: torch.Tensor, std: torch.Tensor, issf_cfg: dict[str, Any]) -> torch.Tensor:
    if bool(issf_cfg.get("include_individual_id", False)):
        raise ValueError("Minimal notebook: include_individual_id not supported")
    P, Cenv = int(env.shape[1]), int(env.shape[0])
    if bool(issf_cfg.get("normalize", True)):
        env = torch.where(valid, (env - mean.view(-1, 1)) / std.view(-1, 1), torch.zeros_like(env))
    move = torch.stack([sl_m, torch.log(sl_m + 1e-8), torch.cos(ta)], dim=0)
    if bool(issf_cfg.get("normalize_movement", True)):
        move[0] = (move[0] - float(mv["sl_mean"])) / float(mv["sl_std"])
        move[1] = (move[1] - float(mv["log_sl_mean"])) / float(mv["log_sl_std"])
        move[2] = (move[2] - float(mv["cos_ta_mean"])) / float(mv["cos_ta_std"])
    parts, valids = [env, move], [valid, torch.ones((3, P), dtype=torch.bool)]
    time = None
    if bool(issf_cfg.get("include_time", False)):
        ts = pd.to_datetime(int(tsn), unit="ns", utc=True)
        doy, hf = int(ts.dayofyear), float(ts.hour) + float(ts.minute) / 60.0 + float(ts.second) / 3600.0
        time = torch.tensor([
            math.sin(2.0 * math.pi * doy / 365.25),
            math.cos(2.0 * math.pi * doy / 365.25),
            math.sin(2.0 * math.pi * hf / 24.0),
            math.cos(2.0 * math.pi * hf / 24.0),
        ], dtype=torch.float32).view(4, 1).expand(4, P)
        parts.append(time)
        valids.append(torch.ones((4, P), dtype=torch.bool))

    def inter(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        Ka, P = a.shape
        Kb = b.shape[0]
        return (a.unsqueeze(1) * b.unsqueeze(0)).reshape(Ka * Kb, P)

    parts.append(inter(move, env))
    valids.append(valid.unsqueeze(0).expand(3, -1, -1).reshape(3 * Cenv, P))
    if time is not None:
        parts.append(inter(time, env))
        valids.append(valid.unsqueeze(0).expand(4, -1, -1).reshape(4 * Cenv, P))
        parts.append(inter(time, move))
        valids.append(torch.ones((12, P), dtype=torch.bool))
    x, v = torch.cat(parts, dim=0), torch.cat(valids, dim=0)
    return torch.where(v, x, torch.zeros_like(x))


summary, surface = [], []
rng = np.random.default_rng(int(SURFACE_SEED))

for study_id in tqdm(studies, desc="studies"):
    study = Study(study_id)
    plan = study.layer_plan(str(layers_config))
    sh = get_sampler_hash(study, str(plan.config_hash))
    mean, std = load_points_normalization_mean_std(study_dir=study.study_dir, plan=plan, sampler_hash=sh)
    mv = mv_stats_from_train_store(study, str(plan.config_hash), sh)

    models = {}
    metas = {}
    for k in ("issf_linear", "issf_mlp", "deepssf"):
        out = load_model(study_id, k)
        if out is not None:
            models[k], metas[k] = out
    if not models:
        continue

    issf_cfg = cfg["datasets"]["issf"]
    s = issf_cfg["sampler"]
    issf_test = study.issf_dataset(
        split="test",
        config=str(layers_config),
        normalize=bool(issf_cfg.get("normalize", True)),
        include_time=bool(issf_cfg.get("include_time", False)),
        include_individual_id=bool(issf_cfg.get("include_individual_id", False)),
        normalize_movement=bool(issf_cfg.get("normalize_movement", True)),
        num_candidates=int(s["num_candidates"]),
        random_seed=int(s["random_seed"]),
        step_multiplier=float(s["step_multiplier"]),
        step_q=float(s["step_q"]),
        gap_q=float(s["gap_q"]),
        bounds_buffer_deg=float(s["bounds_buffer_deg"]),
    )
    issf_loader = DataLoader(issf_test, batch_size=int(issf_cfg.get("batch_size", 32)), shuffle=False, num_workers=int(issf_cfg.get("num_workers", 0) or 0), collate_fn=collate_step_select)
    if "issf_linear" in models:
        r = eval_step_select(models["issf_linear"], issf_loader, DEVICE, topk=TOPK)
        summary.append({"study_id": study_id, "kind": "issf_linear", **r})
    if "issf_mlp" in models:
        r = eval_step_select(models["issf_mlp"], issf_loader, DEVICE, topk=TOPK)
        summary.append({"study_id": study_id, "kind": "issf_mlp", **r})

    dcfg = cfg["datasets"]["deepssf"]
    chips_test = study.chip_dataset(
        split="test",
        config=str(layers_config),
        mode="steps",
        output_hw=tuple(int(x) for x in dcfg.get("output_hw", (101, 101))),
        normalize=bool(dcfg.get("normalize", True)),
        normalization_sampler_hash=sh,
        step_multiplier=float(dcfg.get("step_multiplier", 2.0)),
        step_q=float(dcfg.get("step_q", 0.95)),
        gap_q=float(dcfg.get("gap_q", 0.999)),
    )
    deepssf_test = DeepSSFDataset(chips_test)
    deepssf_loader = DataLoader(deepssf_test, batch_size=int(dcfg.get("batch_size", 32)), shuffle=False, num_workers=int(dcfg.get("num_workers", 0) or 0), collate_fn=collate_deepssf)
    if "deepssf" in models:
        r = eval_deepssf(models["deepssf"], deepssf_loader, DEVICE, topk=TOPK)
        summary.append({"study_id": study_id, "kind": "deepssf", **r})

    df = pd.read_csv(study.trajectories_path)
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[TIME_COL])
    steps = select_step_starts(df[df[SET_COL] == "test"].copy(), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    if len(steps) == 0:
        continue
    n = int(len(steps))
    if SURFACE_STEPS_PER_STUDY is None:
        idxs = list(range(n))
    else:
        k = min(int(SURFACE_STEPS_PER_STUDY), n)
        idxs = sorted(int(x) for x in rng.choice(n, size=k, replace=False).tolist())

    df_tr = compute_movement_features(df[df[SET_COL] == "train"].copy(), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    _sampler, sampler_state, _ = mb_points.build_train_sampler(df_tr, parametric=True, num_candidates=int(ISSF_SURFACE_NUM_CANDIDATES), random_seed=int(s["random_seed"]), step_multiplier=float(s["step_multiplier"]), step_q=float(s["step_q"]), gap_q=float(s["gap_q"]))
    io = study.layer_io(str(layers_config))
    layer_slices = {layer: plan.layer_slice(layer) for layer in plan.layers_in_order}
    px = float(getattr(chips_test, "pixel_size_m"))

    for step_idx in idxs:
        row = steps.iloc[int(step_idx)]

        if ("issf_linear" in models) or ("issf_mlp" in models):
            req = mb_points._StepRequest(
                point_id_start=int(row.point_id_start),
                timestamp=pd.Timestamp(row.timestamp),
                start_lon=float(row.start_lon),
                start_lat=float(row.start_lat),
                start_bearing=float(row.start_bearing),
                end_lon=float(row.end_lon),
                end_lat=float(row.end_lat),
            )
            env_np, valid_np, sl_np, ta_np, pid, tsn = mb_points._extract_one_step(req, io=io, plan=plan, layer_slices=layer_slices, sampler_state=sampler_state, bounds_buffer_deg=float(s["bounds_buffer_deg"]))
            env = torch.from_numpy(env_np).float()[:, :-1]
            valid = torch.from_numpy(valid_np).bool()[:, :-1]
            sl_m = torch.from_numpy(sl_np).float()[:-1]
            ta = torch.from_numpy(ta_np).float()[:-1]
            x_step = issf_step_x(env, valid, sl_m, ta, int(tsn), mv, mean, std, issf_cfg).unsqueeze(0).to(DEVICE)
            headings = float(req.start_bearing) + ta
            dx_all = (sl_m * torch.sin(headings)).numpy()
            dy_all = (sl_m * torch.cos(headings)).numpy()
            for kind in ("issf_linear", "issf_mlp"):
                if kind not in models:
                    continue
                logits = models[kind](x_step).squeeze(0).detach().cpu()
                prob = torch.softmax(logits, dim=0)
                g = torch.Generator(device="cpu")
                g.manual_seed(seed(SURFACE_SEED, study_id, step_idx, kind))
                samp = torch.multinomial(prob, int(ISSF_SURFACE_NUM_SAMPLES), replacement=True, generator=g).numpy()
                for j in samp.tolist():
                    surface.append({"study_id": study_id, "step_idx": int(step_idx), "kind": kind, "dx_m": float(dx_all[int(j)]), "dy_m": float(dy_all[int(j)])})

        if "deepssf" in models:
            spatial, scalars, bearing, _y = deepssf_test[int(step_idx)]
            out = models["deepssf"]((spatial.unsqueeze(0).to(DEVICE), scalars.unsqueeze(0).to(DEVICE), bearing.unsqueeze(0).to(DEVICE)))
            scores = (out[..., 0] + out[..., 1]).squeeze(0)
            H, W = int(scores.shape[0]), int(scores.shape[1])
            prob = torch.softmax(scores.reshape(H * W).detach().cpu(), dim=0)
            g = torch.Generator(device="cpu")
            g.manual_seed(seed(SURFACE_SEED, study_id, step_idx, "deepssf"))
            pix = torch.multinomial(prob, int(DEEPSSF_SURFACE_NUM_SAMPLES), replacement=True, generator=g).numpy()
            rr, cc = (pix // W).astype(int), (pix % W).astype(int)
            dx = (cc - (W // 2)) * px
            dy = ((H // 2) - rr) * px
            for j in range(int(len(dx))):
                surface.append({"study_id": study_id, "step_idx": int(step_idx), "kind": "deepssf", "dx_m": float(dx[j]), "dy_m": float(dy[j])})


df_sum = pd.DataFrame(summary).sort_values(["study_id", "kind"]).reset_index(drop=True)
df_surf = pd.DataFrame(surface)
p_sum, p_surf = out_cmp / "summary.csv", out_cmp / "surface_samples.csv"
df_sum.to_csv(p_sum, index=False)
df_surf.to_csv(p_surf, index=False)
print("Wrote:", p_sum)
print("Wrote:", p_surf)
display(df_sum.head(20))
display(df_surf.head(20))
